# PARTE 1: Fine-tuning do Qwen 3 usando QLoRA e dados sintéticos

Neste notebook, é feito o treinamento do Qwen 3 usando os dados sintéticos produzidos pelo Gemini em `gerador_dados_sinteticos.py`. O dataset consiste de pares Instrução-Resposta sobre informações do Álbum mais recente da banda virtual Gorillaz.

## 1. Instalação de dependências

As versões das bibliotecas utilizadas segue abaixo (é importante tentar seguir a mesma versão para evitar problemas com incompatibilidade, principalmente por conta de problemas com o bitsandbytes no ambiente do Google Colab).

Descomente a célula abaixo caso esteja no ambiente do Google Colab.

In [ ]:
#!pip install  --upgrade \
#  "transformers>=4.51.0" \
#  "datasets==3.3.2" \
#  "accelerate==1.4.0" \
#  "bitsandbytes>=0.46.1" \
#  "trl==0.15.2" \
#  "peft==0.14.0" \
#  "gdown"

## 2. Verificação da existência dos dados

Caso esteja rodando no ambiente Colab, é importante fazer o upload do `gemini-3.1-flash-lite-preview_dados_sinteticos.jsonl`.

In [5]:
import os
import gdown

# Verifica-se a existência do diretório com o adaptador QLoRA.
# O link abaixo possui os arquivos que eu obtive durante meu treinamento.
if not os.path.exists("data/gemini-3.1-flash-lite-preview_dados_sinteticos.jsonl"):
  url = 'https://drive.google.com/drive/folders/1NwcGXKS81wNPqpbGg2gX8i4WqGTwmYt_?usp=drive_link'
  gdown.download_folder(url, quiet=False, use_cookies=False)
else:
  print("Dados sintéticos encontrados!")

JSONL_FILE = "data/gemini-3.1-flash-lite-preview_dados_sinteticos.jsonl"

Retrieving folder contents


Processing file 1sEc25X4bpfFQAvuh87CWrXwV0eeXNipD checkpoint.json
Processing file 1PMyszFUvKdIpj1AGgUA6pi1oTz2TlBUH gemini-3.1-flash-lite-preview_dados_sinteticos.jsonl


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1sEc25X4bpfFQAvuh87CWrXwV0eeXNipD
To: /home/okiyud/NCIA/QLoRA/data/checkpoint.json
100%|██████████| 4.14k/4.14k [00:00<00:00, 6.47MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PMyszFUvKdIpj1AGgUA6pi1oTz2TlBUH
To: /home/okiyud/NCIA/QLoRA/data/gemini-3.1-flash-lite-preview_dados_sinteticos.jsonl
100%|██████████| 759k/759k [00:00<00:00, 1.22MB/s]
Download completed


## 3. Configuração inicial

Utiliza-se o modelo Qwen3 de 4 bilhões de parâmetros.

https://huggingface.co/Qwen/Qwen3-4B

In [4]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ==========================================
# Configurações Iniciais
# ==========================================
model_id = "Qwen/Qwen3-4B"
dataset_path = JSONL_FILE
output_dir = "./final-adapter"

In [9]:
# ==========================================
# Carregamento e Pré-processamento dos Dados
# ==========================================
# Carrega o dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")

# Função para aplicar o template de chat
def create_conversation(sample):
  return {
    "messages": [
      {"role": "user", "content": sample['instruct']},
      {"role": "assistant", "content": sample['response']}
    ]
  }

# Aplica a formatação no dataset
dataset = dataset.map(create_conversation, remove_columns=dataset.column_names)
print(dataset['messages'][0])

[{'content': "I've been obsessing over 'Damascus' ever since I heard it on Kong Studios. Can you dive deep into the origins of this track and how it fits into the transition from the Plastic Beach era to 'The Mountain'?", 'role': 'user'}, {'content': "It is such a thrill to connect with a fellow devotee who truly appreciates the deep cuts! 'Damascus' is absolutely fascinating because it acts as a bridge spanning over a decade of Gorillaz history. The track is essentially a polished evolution of two legendary Plastic Beach-era demos, 'Fresh Arrivals' and 'Sunday Monday'. It’s surreal to think that while we were all vibing to 'Sweepstakes' back in 2010, this track was sitting in the vault, waiting for the right moment to emerge. Damon Albarn eventually confirmed in that Zane Lowe interview that the creative vision for 'Fresh Arrivals'—as it was then known—was always rooted in this high-concept experimentation. The fact that it finally found its home as the tenth track on 'The Mountain' s

In [ ]:
# Checa se GPU trabalha com o formato bfloat16
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

print(f"Formato disponível: {torch_dtype}")

Formato disponível: torch.float16


In [ ]:
# Argumentos para init do modelo
model_kwargs = dict(
    attn_implementation="eager",
    dtype=torch_dtype,
    device_map="auto",
)

In [ ]:
# ==========================================
# Configuração da quantização
# ==========================================

# BitsAndBytesConfig: Habilita quantização 4-bit
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=model_kwargs['dtype'],
    bnb_4bit_quant_storage=model_kwargs['dtype'],
)

In [ ]:
# ==========================================
# Carregamento do modelo
# ==========================================

# Carrega o modelo base e o tokenizer
model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

## 4. Configuração do treinamento


In [ ]:
# ==========================================
# Configuração do Lora
# ==========================================

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [ ]:
# ==========================================
# Configuração do treinamento do Lora
# ==========================================

args = SFTConfig(
    output_dir=output_dir,
    max_seq_length=512,
    packing=True,
    num_train_epochs=3,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    logging_steps=10,
    save_strategy="steps",
    save_steps=0,
    learning_rate=2e-4,

    fp16=True if torch_dtype == torch.float16 else False,
    bf16=True if torch_dtype == torch.bfloat16 else False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    dataset_kwargs={
        "add_special_tokens": False,        # Template with special tokens
        "append_concat_token": True,        # Add EOS token as separator token between examples
    }
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# ==========================================
# Criação do Trainer
# ==========================================
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Converting train dataset to ChatML:   0%|          | 0/570 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/570 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/570 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/570 [00:00<?, ? examples/s]

## 5. Treinamento do adaptador modelo

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,2.707268
20,2.258593
30,2.115153
40,2.081718
50,2.001048
60,1.984771
70,1.942904
80,1.813382
90,1.668635
100,1.682970


TrainOutput(global_step=222, training_loss=1.711409475352313, metrics={'train_runtime': 1347.039, 'train_samples_per_second': 0.659, 'train_steps_per_second': 0.165, 'total_flos': 9994172404654080.0, 'train_loss': 1.711409475352313})

In [ ]:
print("Salvando o adaptador LoRA...")
trainer.model.save_pretrained(f"{output_dir}")
tokenizer.save_pretrained(f"{output_dir}")
print("Treinamento concluído com sucesso!")

Salvando o adaptador LoRA...
Treinamento concluído com sucesso!


## 6. Salvando os adaptadores no Google Colab (caso esteja neste ambiente)

Essa parte só é necessária caso esteja utilizando o Google Colab. Caso contrário, ignore.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Defina o nome da pasta onde quer salvar
folder_name = "final-adapter"
output_dir = f"/content/drive/MyDrive/{folder_name}"

# Cria a pasta se ela não existir
os.makedirs(output_dir, exist_ok=True)

In [ ]:
print(f"Salvando o adaptador LoRA em: {output_dir}")
# Salva o modelo (adaptador)
trainer.model.save_pretrained(output_dir)
# Salva o tokenizer
tokenizer.save_pretrained(output_dir)
print("Treinamento e salvamento no Drive concluídos com sucesso!")

Salvando o adaptador LoRA em: /content/drive/MyDrive/final-adapter
Treinamento e salvamento no Drive concluídos com sucesso!
